In [1]:
import pandas as pd
import numpy as np
import requests

# 🔴 Crypto Risk Monitoring — BTC Anomaly Detection
Detecting anomalous price movements in Bitcoin using z-score analysis.
Data source: CoinGecko API (last 90 days)

## 1. Data Collection
Fetching daily BTC/USD prices from CoinGecko API.

In [2]:
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

params = {
    "vs_currency": "usd",
    "days": "90",
    "interval": "daily"
}

response = requests.get(url, params=params)
data = response.json()

In [3]:
df = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
print(df.head())    

       timestamp         price
0  1772582400000  68321.617848
1  1772668800000  72669.770165
2  1772755200000  70874.986871
3  1772841600000  68148.283400
4  1772928000000  67271.190778


In [4]:
df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df.drop(columns=['timestamp'])
df = df[df['date'].dt.date < pd.Timestamp.today().date()]
print(df.head())

          price       date
0  68321.617848 2026-03-04
1  72669.770165 2026-03-05
2  70874.986871 2026-03-06
3  68148.283400 2026-03-07
4  67271.190778 2026-03-08


In [5]:
df['return'] = df['price'].pct_change() * 100
print(df.head())

          price       date    return
0  68321.617848 2026-03-04       NaN
1  72669.770165 2026-03-05  6.364241
2  70874.986871 2026-03-06 -2.469780
3  68148.283400 2026-03-07 -3.847201
4  67271.190778 2026-03-08 -1.287036


## 2. Anomaly Detection
Calculating daily returns and z-scores to flag abnormal price movements.
- Z-score > 2 or < -2 = anomaly
- Based on 90-day rolling mean and standard deviation

In [6]:
mean = df['return'].mean()
std = df['return'].std()

df['z_score'] = (df['return'] - mean) / std

print(df.head(10))

          price       date    return   z_score
0  68321.617848 2026-03-04       NaN       NaN
1  72669.770165 2026-03-05  6.364241  3.131406
2  70874.986871 2026-03-06 -2.469780 -1.289212
3  68148.283400 2026-03-07 -3.847201 -1.978485
4  67271.190778 2026-03-08 -1.287036 -0.697357
5  66036.157824 2026-03-09 -1.835902 -0.972014
6  68459.315371 2026-03-10  3.669441  1.782905
7  69883.008876 2026-03-11  2.079620  0.987346
8  70226.818791 2026-03-12  0.491979  0.192877
9  70544.425355 2026-03-13  0.452258  0.173001


In [7]:
df['is_anomaly'] = df['z_score'].abs() > 2

In [8]:
print(df[df['is_anomaly'] == True])

           price       date    return   z_score  is_anomaly
1   72669.770165 2026-03-05  6.364241  3.131406        True
20  70892.828240 2026-03-24  4.486371  2.191704        True
35  71975.621710 2026-04-08  4.518147  2.207605        True
41  74514.630067 2026-04-14  5.310983  2.604347        True


## 3. Visualization
Price chart with anomaly flags — red dots mark days with abnormal returns.

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['date'],
    y=df['price'],
    mode='lines',
    name='BTC Price',
    line=dict(color='orange')
))

fig.add_trace(go.Scatter(
    x=df[df['is_anomaly'] == True]['date'],
    y=df[df['is_anomaly'] == True]['price'],
    mode='markers',
    name='Anomaly',
    marker=dict(color='red', size=10)
))

fig.show()

## 4. Rolling Volatility
30-day rolling standard deviation of daily returns.
Higher volatility = higher risk exposure for open positions.

In [10]:
df['volatility'] = df['return'].rolling(window=30).std()
print(df.tail(10))

           price       date    return   z_score  is_anomaly  volatility
79  77546.337653 2026-05-22  0.111533  0.002499       False    1.329402
80  75482.523800 2026-05-23 -2.661394 -1.385097       False    1.342310
81  76672.792891 2026-05-24  1.576880  0.735771       False    1.376932
82  76988.161261 2026-05-25  0.411317  0.152513       False    1.366676
83  77274.401522 2026-05-26  0.371798  0.132737       False    1.367811
84  75824.063325 2026-05-27 -1.876868 -0.992513       False    1.385313
85  74352.699280 2026-05-28 -1.940497 -1.024354       False    1.398052
86  73539.841896 2026-05-29 -1.093245 -0.600382       False    1.392151
87  73382.718314 2026-05-30 -0.213658 -0.160229       False    1.387202
88  73751.067873 2026-05-31  0.501957  0.197870       False    1.384231
